# Leitura dos Dados

In [2]:
import pandas as pd
arquivos = [
    'dim_clientes',  
    'dim_geografia',  
    'dim_produtos',
    'dim_vendedores',  
    'fact_avaliações',
    'fact_itens',
    'fact_pagamentos',            
    'fact_pedidos',
]   
dado_cru = {}        
for nome in arquivos:
    dado_cru[nome] = pd.read_csv(rf"C:\Users\daniel-pc\Documents\Projetos\Analise Exploratória E-Commerce\{nome}.csv")
# Atribuição de variáveis
clientes = dado_cru['dim_clientes']
geografia = dado_cru['dim_geografia']
produtos = dado_cru['dim_produtos']         
vendedores = dado_cru['dim_vendedores']      
avaliacoes = dado_cru['fact_avaliações']      
itens = dado_cru['fact_itens']      
pagamentos = dado_cru['fact_pagamentos']
pedidos = dado_cru['fact_pedidos']      



#### Investigação e limpeza da tabela clientes

In [ ]:
print('Dados Crus:')
print(clientes.head(5))
# Mapeia 20 principais cidades do Brasil (20/80 [Principio de Pareto])
mapa_pareto_esmagado = {
    'saopaulo': 'São Paulo',
    'riodejaneiro': 'Rio de Janeiro',
    'belohorizonte': 'Belo Horizonte',
    'brasilia': 'Brasília',
    'curitiba': 'Curitiba',
    'campinas': 'Campinas',
    'portoalegre': 'Porto Alegre',
    'salvador': 'Salvador',
    'guarulhos': 'Guarulhos',
    'saobernardocampo': 'São Bernardo do Campo',
    'niteroi': 'Niterói',
    'santoandre': 'Santo André',
    'osasco': 'Osasco',
    'goiania': 'Goiânia',
    'saojosedoscampos': 'São José dos Campos',
    'recife': 'Recife',
    'fortaleza': 'Fortaleza',
    'sorocaba': 'Sorocaba',
    'ribeiraopreto': 'Ribeirão Preto',
    'florianopolis': 'Florianópolis'
}

def padronizar_cidade_br(nome_cru):
    texto_limpo = str(nome_cru).lower().strip()
    
    # Junta tudo sem espaço nenhum
    # Exemplo: " sao   paulo " -> split vira ['sao', 'paulo'] -> join vira "saopaulo"
    digital = ''.join(texto_limpo.split())
    
    # 1. Consulta pela digital no VIP
    if digital in mapa_pareto_esmagado:
        return mapa_pareto_esmagado[digital]
    
    # 2. Se não achou no VIP, aplica a regra gramatical normal usando o texto original
    preposicoes_br = {'de', 'da', 'do', 'das', 'dos', 'd', 'e', 'del'}
    palavras = texto_limpo.split()
    
    formatadas = []
    for i, palavra in enumerate(palavras):
        if i > 0 and palavra in preposicoes_br:
            formatadas.append(palavra)
        else:
            formatadas.append(palavra.capitalize())
            
    return ' '.join(formatadas)

clientes['customer_city'] = clientes['customer_city'].apply(padronizar_cidade_br)

print('Dados Transformados:')
print(clientes['customer_city'].value_counts().head(10))


Dados Crus:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 Franca             SP  
1                      9790  Sao Bernardo do Campo             SP  
2                      1151              São Paulo             SP  
3                      8775        Mogi das Cruzes             SP  
4                     13056               Campinas             SP  
Dados Transformados:
customer_city
São Paulo                15540
Rio de Janeiro            6882
Belo Horizonte            2773
Brasília     

#### Investigação e limpeza da tabela geografia

In [ ]:
print(f"Linhas originais brutas: {len(geografia)}")
geo_sem_clones = geografia.drop_duplicates()

# COMPRESSÃO (Pega coordenadas de um mesmo CEP e agrupa pela media de cordenadas)
dim_geo_compressa = geo_sem_clones.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print(f"Linhas após compressão: {len(dim_geo_compressa)}")

# Aplica função na tabela COMPRIMIDA
dim_geo_compressa['geolocation_city'] = dim_geo_compressa['geolocation_city'].apply(padronizar_cidade_br)



Linhas originais brutas: 1000163
Linhas após compressão: 19015


#### Investigação e limpeza da tabela produtos

In [ ]:
print(produtos.info())
print(f"Duplicatas: {produtos.duplicated().sum()}")
produtos.head()

# CORREÇÃO GRAMATICAL DO INGLÊS (Consertando o "lenght")
produtos = produtos.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)

# Tratamento de produtos com categoria vazia
produtos['product_category_name'] = produtos['product_category_name'].fillna("outros")

# Tratamento de fotos (Faltam 610 também. Quem não tem categoria, não tem foto)
# Preenche nulos com 0 e transforma de Float (2.0) para Inteiro (2)
produtos["product_photos_qty"] = (
    produtos["product_photos_qty"].fillna(0).astype(int)
)

# Cria dicionario para repetição futura
colunas_fisicas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for col in colunas_fisicas:
    mediana_segura = produtos[col].median() # Pega mediana para todas as colunas fisicas 
    produtos[col] = produtos[col].fillna(mediana_segura) # Preenche nulos das colunas com mediana
produtos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None
Duplicatas: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                  

#### Investigação e limpeza da tabela vendedores

In [ ]:
vendedores.head()
vendedores['seller_city'] = vendedores['seller_city'].apply(padronizar_cidade_br)
vendedores.head()

caminho_salvar = rf"C:\Users\daniel-pc\Documents\Projetos\Analise Exploratória E-Commerce\dados_limpos\dim_vendedores_limpa.csv"

#### Investigação e limpeza da tabela avaliações


In [ ]:
# Conversão de nulos dos comentários para "Nenhum"
avaliacoes["review_comment_title"] = avaliacoes["review_comment_title"].fillna(
    "Nenhum"
)
avaliacoes["review_comment_message"] = avaliacoes[
    "review_comment_message"
].fillna("Nenhum")

# 2. Conversão de tipo objeto para tipo data
avaliacoes["review_creation_date"] = pd.to_datetime(
    avaliacoes["review_creation_date"]
)
avaliacoes["review_answer_timestamp"] = pd.to_datetime(
    avaliacoes["review_answer_timestamp"]
)

avaliacoes.info()
avaliacoes.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     99224 non-null  object        
 4   review_comment_message   99224 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


#### Investigação e limpeza da tabela itens

In [ ]:
itens.info()
# .str.replace('"', '') -> Arranca as aspas duplas
# .str.strip()          -> Arranca os espaços invisíveis do começo e do fim
itens.columns = itens.columns.str.replace('"', '').str.strip()
itens["shipping_limit_date"] = pd.to_datetime(itens["shipping_limit_date"])
itens.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   order_id                             112650 non-null  object 
 1    "order_item_id"                     112650 non-null  int64  
 2    "product_id"                        112650 non-null  object 
 3    "seller_id"                         112650 non-null  object 
 4    "shipping_limit_date"               112650 non-null  object 
 5    "price"                             112650 non-null  float64
 6    "freight_value"                     112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         

#### Investigação e limpeza da tabela pagamentos


In [ ]:
# VALORES NULOS (Dinheiro sumido)
print("1. Valores nulos por coluna:")
print(pagamentos.isnull().sum())

# O que os clientes usaram para pagar?
print("\n2. Contagem por Método de Pagamento:")
print(pagamentos['payment_type'].value_counts())

# Alguém pagou em 0x ou em número negativo?)
parcelas_estranhas = pagamentos[pagamentos['payment_installments'] < 1]
print(f"\n3. Pagamentos com menos de 1 parcela: {len(parcelas_estranhas)} linhas")

# VALORES PAGOS ZERADOS OU NEGATIVOS
pagamento_gratis = pagamentos[pagamentos['payment_value'] <= 0]
print(f"4. Pagamentos com valor R$ 0,00 ou negativo: {len(pagamento_gratis)} linhas")


# Elimina as 3 transações fantasmas 'not_defined'
pagamentos_limpos = pagamentos[pagamentos['payment_type'] != 'not_defined'].copy()

# Corrige a aberração matemática de "0 parcelas" para à vista (1x)
pagamentos_limpos.loc[pagamentos_limpos['payment_installments'] == 0, 'payment_installments'] = 1

# Garante que as parcelas sejam inteiras (int)
pagamentos_limpos['payment_installments'] = pagamentos_limpos['payment_installments'].astype(int)

print(f"Linhas antes: {len(pagamentos)} | Linhas depois: {len(pagamentos_limpos)}")


1. Valores nulos por coluna:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

2. Contagem por Método de Pagamento:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

3. Pagamentos com menos de 1 parcela: 2 linhas
4. Pagamentos com valor R$ 0,00 ou negativo: 9 linhas
Linhas antes: 103886 | Linhas depois: 103883


#### Investigação, limpeza e criação de métrica (lead time) na tabela pedidos

In [ ]:
import requests
import numpy as np
print(f"Contagem inicial de nulos:\n{pedidos.isnull().sum()}")
print("-" * 60)

# 2. CONVERSÃO EM MASSA PARA DATETIME OFICIAL
colunas_datas = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in colunas_datas:
    pedidos[col] = pd.to_datetime(pedidos[col])


# 3. EXTRAÇÃO DE FERIADOS VIA API (BrasilAPI)
print("Conectando à BrasilAPI para buscar feriados nacionais (2016-2018)...")
lista_feriados = []

for ano in [2016, 2017, 2018]:
    url = f"https://brasilapi.com.br/api/feriados/v1/{ano}"
    resposta = requests.get(url)

    if resposta.status_code == 200:
        dados_feriados = resposta.json()
        for feriado in dados_feriados:
            lista_feriados.append(feriado["date"])

# Converte a lista de strings para o formato de data estrita do NumPy
feriados_finais = np.array(lista_feriados, dtype="datetime64[D]")
print(f"-> {len(feriados_finais)} feriados nacionais mapeados")
print("-" * 60)


# 4. Tratando os nulos falsos
# Entregues sem data de APROVAÇÃO (Herda a data da compra)
mascara_entregue_sem_aprovacao = (pedidos["order_status"] == "delivered") & (
    pedidos["order_approved_at"].isnull()
)
pedidos.loc[mascara_entregue_sem_aprovacao, "order_approved_at"] = (
    pedidos.loc[mascara_entregue_sem_aprovacao, "order_purchase_timestamp"]
)

# Entregues sem data de ENTREGA (Herda a data estimada)
mascara_entregue_sem_data = (pedidos["order_status"] == "delivered") & (
    pedidos["order_delivered_customer_date"].isnull()
)
pedidos.loc[mascara_entregue_sem_data, "order_delivered_customer_date"] = (
    pedidos.loc[mascara_entregue_sem_data, "order_estimated_delivery_date"]
)


# 5. CÁLCULO VETORIZADO DO LEAD TIME (Dias Úteis desatando Finais de Semana e Feriados)

# Cria a máscara para processar APENAS pedidos que de fato foram entregues
mask_entregues = pedidos["order_delivered_customer_date"].notnull()

# Inicializa a nova coluna preenchida com nulo padrão
pedidos["prazo_dias_uteis"] = np.nan

# Isola e extrai os arrays de data pura [D] para o NumPy processar
datas_inicio = (
    pedidos.loc[mask_entregues, "order_purchase_timestamp"]
    .dt.date.values.astype("datetime64[D]")
)
datas_fim = (
    pedidos.loc[mask_entregues, "order_delivered_customer_date"]
    .dt.date.values.astype("datetime64[D]")
)

# O motor executa a contagem saltando finais de semana e cortando os feriados da API
pedidos.loc[mask_entregues, "prazo_dias_uteis"] = np.busday_count(
    datas_inicio, datas_fim, holidays=feriados_finais
)

# Transforma em Int64 (com I maiúsculo) para aceitar inteiros e NaNs na mesma coluna
pedidos["prazo_dias_uteis"] = pedidos["prazo_dias_uteis"].astype("Int64")


# 6. AUDITORIA FINAL DE QUALIDADE
print("-" * 60)
print(f"Contagem final de nulos pós-tratamento:\n{pedidos.isnull().sum()}")
print("-" * 60)
print("Amostra dos resultados calculados:")
print(
    pedidos[mask_entregues][
        [
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "prazo_dias_uteis",
        ]
    ].head(5)
)

Contagem inicial de nulos:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
------------------------------------------------------------
Conectando à BrasilAPI para buscar feriados nacionais (2016-2018)...
-> 36 feriados nacionais mapeados com sucesso!
------------------------------------------------------------
Calculando prazos reais de entrega em Dias Úteis...
------------------------------------------------------------
Contagem final de nulos pós-tratamento:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 146
order_delivered_carrier_date     1783
order_delivered_customer_date    2957

##### Atrelação de IPCA à data do pedido

In [ ]:
# 2. CONSUMINDO A API DO BANCO CENTRAL (Série 433: IPCA Mensal)
print("Conectando ao Banco Central do Brasil (SGS)...")
# Parâmetros: Formato JSON, Data Inicial 01/01/2016, Data Final 31/12/2018
url_bcb = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2016&dataFinal=31/12/2018"

resposta = requests.get(url_bcb)
if resposta.status_code == 200:
    dados_bcb = resposta.json()
    # Transformamos o JSON em uma tabela (DataFrame)
    ipca = pd.DataFrame(dados_bcb)
    print("-> Sucesso! Série histórica do IPCA baixada.")
else:
    print(f"Erro na API do Banco Central. Código: {resposta.status_code}")

    
# 3. TRATANDO OS DADOS DO BANCO CENTRAL
# A data vem como 'dd/mm/aaaa' e o valor vem como texto. Precisamos limpar.
ipca['data'] = pd.to_datetime(ipca['data'], format='%d/%m/%Y')
ipca['taxa_ipca_mensal'] = ipca['valor'].astype(float)

# Criamos a chave de ligação (Ex: '2017-04')
ipca['ano_mes'] = ipca['data'].dt.to_period('M')
# Guardamos só o que importa
ipca_limpo = ipca[['ano_mes', 'taxa_ipca_mensal']]


# 4. PREPARANDO A TABELA DE PEDIDOS PARA O MERGE
# Extraímos o Ano-Mês da data exata em que o cliente passou o cartão
pedidos['ano_mes'] = pedidos['order_purchase_timestamp'].dt.to_period('M')


# 5. O GRANDE MERGE (O Casamento dos Dados)
# Colamos a taxa de inflação na linha de cada pedido
pedidos_enriquecidos = pd.merge(pedidos, ipca_limpo, on='ano_mes', how='left')

# Excluímos a coluna 'ano_mes' pois ela já fez o trabalho dela
pedidos_enriquecidos = pedidos_enriquecidos.drop(columns=['ano_mes'])


# 6. AUDITORIA E SALVAMENTO
print("\nAmostra dos Pedidos com a Inflação (IPCA) do mês atrelada:")
print(pedidos_enriquecidos[['order_id', 'order_purchase_timestamp', 'taxa_ipca_mensal']].head())

Conectando ao Banco Central do Brasil (SGS)...
-> Sucesso! Série histórica do IPCA baixada.

Amostra dos Pedidos com a Inflação (IPCA) do mês atrelada:
                           order_id order_purchase_timestamp  taxa_ipca_mensal
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33              0.42
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37              0.33
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49             -0.09
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:06              0.28
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:39              0.32


### Salvamento em formato parquet para injeção em data lake

In [ ]:
from pathlib import Path

pasta_destino = Path(r"C:\Users\daniel-pc\Documents\Projetos\Analise Exploratória E-Commerce\dados_limpos_parquet")

tabelas_para_salvar = {
    "dim_clientes": clientes,
    "dim_geografia": dim_geo_compressa,
    "dim_produtos": produtos,
    "fact_itens": itens,
    "fact_avaliacoes": avaliacoes,
    "fact_pagamentos": pagamentos_limpos,
    "fact_pedidos": pedidos_enriquecidos     
}


# EXPORTAÇÃO
print(f"Diretório alvo: {pasta_destino}\n")

for nome_arquivo, dataframe in tabelas_para_salvar.items():
    
    # Monta o caminho final. Ex: ...\dados_limpos_parquet\fact_pedidos.parquet
    caminho_arquivo = pasta_destino / f"{nome_arquivo}.parquet"
    
    dataframe.to_parquet(caminho_arquivo, engine='pyarrow', index=False)
    
    # progresso no terminal
    tamanho_mb = caminho_arquivo.stat().st_size / (1024 * 1024)
    print(f"{nome_arquivo}.parquet gerado com sucesso! ({tamanho_mb:.2f} MB)")

--- INICIANDO MIGRAÇÃO PARA FORMATO PARQUET ---
Diretório alvo: C:\Users\daniel-pc\Documents\Projetos\Analise Exploratória E-Commerce\dados_limpos_parquet

dim_clientes.parquet gerado com sucesso! (6.68 MB)
dim_geografia.parquet gerado com sucesso! (0.54 MB)
dim_produtos.parquet gerado com sucesso! (1.34 MB)
fact_itens.parquet gerado com sucesso! (6.37 MB)
fact_avaliacoes.parquet gerado com sucesso! (9.16 MB)
fact_pagamentos.parquet gerado com sucesso! (3.65 MB)
fact_pedidos.parquet gerado com sucesso! (10.06 MB)
